# Extracting the full conductivity tensor in a rectangular sample

Worked example for the [`rectcond`](https://github.com/jag47/Extracting-the-full-conductivity-tensor-in-a-rectangular-sample) package, which implements

> Julia Gelfond and Oskar Vafek, *Extracting the full conductivity tensor in a rectangular sample*, [arXiv:2608.06643](https://arxiv.org/abs/2608.06643) (2026).

Four four-terminal resistance measurements on a rectangular sample determine the full 2D conductivity tensor: the principal conductivities $\sigma_+ \geq \sigma_-$, the Hall conductivity $\sigma_H$, and the principal-axes angle $\alpha$.

Install the package first (see the repository README), e.g.

```sh
pip install git+https://github.com/jag47/Extracting-the-full-conductivity-tensor-in-a-rectangular-sample.git
```

## Inputs

The sample is a rectangle with side $d_1$ along $x$ and side $d_2$ along $y$. The four resistances $R_1, R_2, R_3, R_5$ are measured in the contact configurations of Fig. 1f–h of the paper (see the README for the exact source/drain/probe placements and sign conventions — note $R_3$ may legitimately be negative).

The values below are the synthetic test case shipped with the original release scripts.

In [1]:
from rectcond import extract_tensor

d1, d2 = 2.7, 1.2

R1 = 0.1636153846153846
R2 = 0.01676923076923077
R3 = -0.055923076923076916
R5 = 0.04815384615384615

## Sanity checks before extracting

Two quick consistency checks from the paper: $R_2 < R_5$ must hold (otherwise an assumption such as spatial uniformity of the tensor is violated), and if the optional fourth vertex configuration was measured, it should satisfy $R_4 = -2R_1 + 2R_2 - R_3$ (Eq. 39).

In [2]:
assert R2 < R5, "R2 >= R5: measurement inconsistent with a uniform conductivity tensor"

R4_predicted = -2 * R1 + 2 * R2 - R3
print(f"if you also measured configuration 4, expect R4 = {R4_predicted:.6g}")

if you also measured configuration 4, expect R4 = -0.237769


## Extraction

In [3]:
result = extract_tensor(d1, d2, R1, R2, R3, R5)

print(f"sigma_+  = {result.sigma_plus:.6f}")
print(f"sigma_-  = {result.sigma_minus:.6f}")
print(f"sigma_H  = {result.sigma_hall:.6f}")
print(f"alpha    = {result.alpha:.6f} rad")
print()
print(f"geometric mean sqrt(sigma_+ sigma_-) = {result.geometric_mean:.6f}")
print(f"anisotropy ratio sqrt(sigma_-/sigma_+) = {result.anisotropy_ratio:.6f}")
print()
print("intermediate (diagnostic) parameters:")
print(f"  r  = {result.r:.6f}")
print(f"  a  = {result.a:.6f}")
print(f"  z5 = {result.z5:.6f}")

sigma_+  = 4.999174
sigma_-  = 1.999138
sigma_H  = 0.999526
alpha    = 3.141368 rad

geometric mean sqrt(sigma_+ sigma_-) = 3.161335
anisotropy ratio sqrt(sigma_-/sigma_+) = 0.632371

intermediate (diagnostic) parameters:
  r  = 0.912473
  a  = 0.500068
  z5 = 0.709570


## Interpretation

Conductivities are sheet conductances in units of $1/[R]$ ([S/sq] for resistances in ohms). $\alpha \in [0, \pi)$ is measured counterclockwise from the bottom ($d_1$) edge; it is the orientation of one principal axis, with the other at $\alpha + \pi/2$.

The recovered values here are consistent with the round numbers $\sigma_+ = 5$, $\sigma_- = 2$, $\sigma_H = 1$ with principal axes aligned to the sample edges ($\alpha \equiv 0 \bmod \pi$), confirming the round trip on this synthetic test case. The parameter $a$ is close to its special value $1/2$, which is exactly the aligned-axes case ($a = 1/2 \Rightarrow \alpha = 0$ or $\pi/2$; see Eq. 18 of the paper).

In [4]:
import numpy as np

expected = {"sigma_+": 5.0, "sigma_-": 2.0, "sigma_H": 1.0}
recovered = {"sigma_+": result.sigma_plus, "sigma_-": result.sigma_minus, "sigma_H": result.sigma_hall}

print(f"{'quantity':<10} {'expected':>10} {'recovered':>12} {'rel. error':>12}")
for key in expected:
    rel = abs(recovered[key] - expected[key]) / expected[key]
    print(f"{key:<10} {expected[key]:>10.4f} {recovered[key]:>12.6f} {rel:>12.2e}")

alpha_mod = result.alpha % np.pi
alpha_dev = min(alpha_mod, np.pi - alpha_mod)
print(f"\nalpha deviation from 0 (mod pi): {alpha_dev:.2e} rad")

quantity     expected    recovered   rel. error
sigma_+        5.0000     4.999174     1.65e-04
sigma_-        2.0000     1.999138     4.31e-04
sigma_H        1.0000     0.999526     4.74e-04

alpha deviation from 0 (mod pi): 2.25e-04 rad
